# Lint your Arize Phoenix agent traces with tracelint

You already collect your agent's traces in **Phoenix**. [**tracelint**](https://github.com/AshwinUgale/tracelint) runs *on top of* them: it reads the spans you have and reports **structural** defects — ignored tool errors, schema-violating calls, hallucinated arguments, loops, **duplicate side effects** — each with the exact span as evidence and a CI exit code.

**No second model judges the trace.** For this class of bug a judge is the wrong tool, because the defect is *decidable by looking at the trace*. Deterministic, reproducible, and ~free.

## Install

```bash
pip install tracelint
```

## On your live Phoenix traces

Phoenix hands you spans as a dataframe, and tracelint reads that exact shape. Export once:

In [ ]:
import json
import phoenix as px

records = px.Client().get_spans_dataframe().to_dict("records")  # your real agent traces
json.dump(records, open("spans.json", "w"))

Then lint them — one report per trace:

In [ ]:
from tracelint import ToolRegistry, load_source, default_rules, lint_trace, render_report

registry = ToolRegistry.load("tools.json")   # optional: declares side_effecting / failure_when
for trace in load_source("spans.json", "openinference"):
    print(render_report(lint_trace(trace, default_rules(), registry), include_candidates=True))

## Run it now — offline, no Phoenix instance or API key

This is an **illustrative** Phoenix-shape trace (constructed, not a captured run) — swap in your own `spans.json` export from the cells above to lint real traces. Here the support agent is asked to refund order `A100`, the `get_order` lookup **errors**, and the agent refunds the card anyway — using the errored order id — and refunds it **twice**.

In [ ]:
import json
from tracelint import ToolRegistry, lint_otel_trace, render_report


def tool_span(sid, start, name, args, output, error=None):
    s = {"span_id": sid, "trace_id": "support-run", "start_time": start, "name": name,
         "status_code": "ERROR" if error else "OK",
         "attributes": {"openinference.span.kind": "TOOL", "tool.name": name,
                        "input.value": json.dumps(args), "output.value": json.dumps(output)}}
    if error:
        s["status_message"] = error
    return s


spans = [
    # The opening LLM span carries what the model was asked, so provenance (R3) sees the order id
    # came from the user and does not flag it.
    {"span_id": "s0", "trace_id": "support-run", "start_time": "2024-06-01T10:00:00Z",
     "name": "agent", "status_code": "OK",
     "attributes": {"openinference.span.kind": "LLM",
                    "llm.input_messages.0.message.role": "user",
                    "llm.input_messages.0.message.content": "Refund order A100 to the card on file."}},
    tool_span("s1", "2024-06-01T10:00:01Z", "get_order", {"order_id": "A100"},
              {"order_id": "A100", "status": "error"}, error="500 internal error"),
    tool_span("s2", "2024-06-01T10:00:02Z", "refund_order", {"order_id": "A100"}, {"refunded": True}),
    tool_span("s3", "2024-06-01T10:00:03Z", "refund_order", {"order_id": "A100"}, {"refunded": True}),
]

# The operator's tools.json: refund_order mutates the world, and {"refunded": false} is its
# declared failure. Declared once, never guessed from the tool name.
registry = ToolRegistry.from_dict({"tools": {
    "get_order": {},
    "refund_order": {"metadata": {"side_effecting": True,
                                  "failure_when": {"pointer": "/refunded", "equals": False}}}}})

report = lint_otel_trace(spans, registry=registry)
print(render_report(report, include_candidates=True))
print("\nexit code:", report.exit_code, " (2 = a hard defect; fails CI)")

You'll see three findings, each pointing at the exact spans:

- **R2a `hard_event`** — `get_order` returned an error (read straight from the OTel `ERROR` status).
- **R2b `hard_defect`** — the errored order id was **reused** as the argument to a side-effecting `refund_order`. Data from a failed call fed into a real-world action, no fallback. This is the tier that **fails CI** (exit `2`).
- **R8 `hard_event`** — `refund_order` was called **twice** with the same arguments after the first succeeded: a **double refund**.

It also discloses what it *couldn't* check (no schema for these tools → R1 suppressed) plus per-rule **verification coverage**, so a clean report is honestly clean — not just empty.

## In CI

```bash
tracelint check spans.json --format openinference --tools tools.json
```

Exit `2` on a hard defect fails the build. Or drop in the GitHub Action:

```yaml
- uses: AshwinUgale/tracelint@v0.5.0
  with:
    path: spans.json
    format: openinference
    tools: tools.json
```

Repo & docs: **https://github.com/AshwinUgale/tracelint**